**1. Install Libraries**

In [ ]:
!pip install -U "lm_eval[hf]"

# import librarys
import os
import lm_eval
from lm_eval.tasks import TaskManager
import zipfile
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0db83afa5c561466c21c8389af96a7a55d8a14d67d256d32ccbcf3c7bc3239bf
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=841f8c9fffcdcd4d2dbe38557898fd75becf0829985b4eef28df633aa9723ffa
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21/

**2. Load HF Token**

In [ ]:
os.environ["HF_TOKEN"] = "add key"

**3. Examine Dataset**

In [ ]:
# load the task
task_manager = TaskManager()
task_dict = lm_eval.tasks.get_task_dict(["medqa_4options"], task_manager)
task = task_dict["medqa_4options"]

# build the dataset
task.build_all_requests(rank=0, world_size=1)

# print dataset size
test_docs = list(task.test_docs())
print(f"Test set size: {len(test_docs)}")

# print first example prompt
print("\n" + "="*50)
print("EXAMPLE PROMPT:")
print("="*50)
example = test_docs[0]
print(task.doc_to_text(example))
print("\n" + "="*50)
print("CORRECT ANSWER:")
print("="*50)
print(task.doc_to_target(example))

README.md:   0%|          | 0.00/640 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1272 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

100%|██████████| 1273/1273 [00:00<00:00, 65459.67it/s]

Test set size: 1273

EXAMPLE PROMPT:
Question: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report
Answer:

CORRECT ANSWER:
1


**4. Load LoRA Adaptar Llama-3.2-3B-Instruct (Medical + Math LoRA) Convert From Zip**

In [ ]:
# unzip the lora adapter
zip_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Low-Rank-Adaptation-(LoRA)/Llama-3.2-3B-Instruct/Medical-Math/medical-math_llama3_lora_adapter.zip"
extract_path = "/content/llama_medical_math_lora_adapter"

os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzipped!")
print(f"Adapter location: {extract_path}")
print(os.listdir(extract_path))

Unzipped!
Adapter location: /content/llama_medical_math_lora_adapter
['medical-math_llama3_lora_adapter']


**5. Load Base Model and Apply Adaptar**

In [ ]:
# path to the unzipped lora adapter
adapter_path = "/content/llama_medical_math_lora_adapter/medical-math_llama3_lora_adapter"

# base model name from hugging face
base_model_name = "meta-llama/Llama-3.2-3B-Instruct"

# loads the tokenizer from the base model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# loads the base model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

# loads the lora adapter on top of the base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_path)

# merge and unload for lm-eval
print("Merging LoRA adapter...")
model = model.merge_and_unload()

# save merged model
print("Saving merged model...")
model.save_pretrained("/content/llama_medical_math_lora")
tokenizer.save_pretrained("/content/llama_medical_math_lora")

# confirm paths and files
print(f"\nAdapter path:      {adapter_path}")
print(f"Merged model path: /content/llama_medical_math_lora")
print(f"Files:             {os.listdir('/content/llama_medical_math_lora')}")
print("Ready for lm-eval!")

Loading tokenizer...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loading LoRA adapter...
Merging LoRA adapter...
Saving merged model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Adapter path:      /content/llama_medical_math_lora_adapter/medical-math_llama3_lora_adapter
Merged model path: /content/llama_medical_math_lora
Files:             ['model.safetensors', 'tokenizer.json', 'config.json', 'chat_template.jinja', 'tokenizer_config.json', 'generation_config.json']
Ready for lm-eval!


**6. Perform Evaluation (medqa_4options)**

In [ ]:
!lm-eval run \
  --model hf \
  --model_args pretrained=/content/llama_medical_math_lora \
  --tasks medqa_4options \
  --device cuda:0 \
  --batch_size 8 \
  --apply_chat_template \
  --output_path /content/drive/MyDrive/Model-Evaluations/MedQA-USMLE-4-options/Llama-3.2-3B-Instruct-LoRA-Models/llama_medical_math_lora

2026-04-22:21:35:41 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-22:21:35:46 INFO     [_cli.run:376] Selected Tasks: ['medqa_4options']
2026-04-22:21:35:48 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-22:21:35:48 INFO     [evaluator:236] Initializing hf model, with arguments: {'pretrained': '/content/llama_medical_math_lora'}
2026-04-22:21:35:52 INFO     [models.huggingface:161] Using device 'cuda:0'
2026-04-22:21:35:54 INFO     [models.huggingface:423] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
Loading weights: 100% 254/254 [00:01<00:00, 158.83it/s, Materializing param=model.norm.weight]
2026-04-22:21:35:58 INFO     [tasks:700] Selected tasks:
2026-04-22:21:35:58 INFO     [tasks:691] Task: medqa_4options (medqa/medqa.yaml)
2026-04-22:21:35:58 WARNING  [evaluator:490] Chat tem